# NB02 · Silver (Spain Utilities Pulse)

Lee todos los JSON de `Files/bronze/<fuente>/`, tipa y valida, y hace `MERGE` en tablas Delta del esquema `silver`.

- **Gana el último valor descargado** para cada clave natural. Por eso Bronze solapa días: si una fuente corrige un dato, la corrección actualiza la fila existente.
- **Solo se escribe lo que cambia:** `MERGE` inserta claves nuevas y actualiza únicamente las filas cuyo valor ha cambiado. Volver a ejecutar no duplica nada.
- **Días incompletos:** REE y ENTSOG solo aceptan días con al menos 2 días de antigüedad.
- **Valores no válidos:** se descartan filas sin fecha, sin clave o sin valor, y valores negativos (o cero en la demanda eléctrica).
- **Números de MITECO:** llegan como texto con coma decimal (`'91,00'`) y se convierten a número.
- Particiones por `anio` y `mes`, y columna `cargado_en` con la fecha de carga.

| Tabla Silver | Clave natural | Origen |
|---|---|---|
| `silver.ree_demanda_diaria` | `serie`, `fecha` | REE |
| `silver.entsog_flujo_fisico` | `fecha`, `point_key`, `direction_key` | ENTSOG |
| `silver.agsi_almacenamiento` | `fecha` | GIE AGSI+ |
| `silver.miteco_embalses_semanal` | `ambito`, `embalse`, `fecha` | MITECO |

**Validaciones al final:** las tablas de REE, ENTSOG y AGSI+ no pueden estar vacías ni tener claves repetidas. Si algo falla, el notebook lanza un error y el pipeline no ejecuta Gold.

Requiere un Lakehouse con **esquemas habilitados** y asociado como predeterminado.

**Resultado de la ejecución del 25 nov 2026**

| Tabla | Descartadas | Insertadas | Actualizadas | Filas en la tabla |
|---|---|---|---|---|
| `silver.ree_demanda_diaria` | 2 | 2 | 1 | 1.058 |
| `silver.entsog_flujo_fisico` | 1.854 | 34 | 0 | 19.214 |
| `silver.agsi_almacenamiento` | 0 | 2 | 3 | 1.058 |
| `silver.miteco_embalses_semanal` | 0 | 245.421 | 0 | 245.421 |

In [ ]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
import datetime as dt

SCHEMA = "silver"
BRONZE = "Files/bronze"
HOY = dt.date.today()
LIMITE_REE = HOY - dt.timedelta(days=2)      # días con >= 2 días de antigüedad
LIMITE_ENTSOG = HOY - dt.timedelta(days=2)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
resumen = []
print(f"Hoy: {HOY} | REE/ENTSOG hasta {LIMITE_REE}")

In [ ]:
def leer_bronze(fuente):
    # Cada JSON tiene {meta, data}; descargado (meta) ordena qué versión del dato es la más reciente
    return (spark.read.option("multiLine", True).json(f"{BRONZE}/{fuente}/*.json")
            .select(F.col("meta.descargado").alias("descargado"), F.col("data")))

def ultimo_valor(df, claves):
    w = Window.partitionBy(*claves).orderBy(F.col("descargado").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "descargado")

def con_particion(df):
    return (df.withColumn("anio", F.year("fecha")).withColumn("mes", F.month("fecha"))
              .withColumn("cargado_en", F.current_timestamp()))

def upsert(df, tabla, claves, fuente, descartadas):
    full = f"{SCHEMA}.{tabla}"
    valores = [c for c in df.columns if c not in claves + ["anio", "mes", "cargado_en"]]
    if not spark.catalog.tableExists(full):
        df.write.format("delta").partitionBy("anio", "mes").saveAsTable(full)
        ins, upd = df.count(), 0
    else:
        cond = " AND ".join(f"t.{k} = s.{k}" for k in claves)
        cambio = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in valores)
        (DeltaTable.forName(spark, full).alias("t").merge(df.alias("s"), cond)
            .whenMatchedUpdateAll(condition=cambio).whenNotMatchedInsertAll().execute())
        m = DeltaTable.forName(spark, full).history(1).select("operationMetrics").first()[0]
        ins, upd = int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))
    resumen.append({"tabla": full, "claves": ", ".join(claves), "descartadas": descartadas,
                    "insertadas": ins, "actualizadas": upd, "filas_tabla": spark.table(full).count()})

In [ ]:
# REE: demanda diaria. data = lista de respuestas; cada una trae series (included) con values diarios
raw = leer_bronze("ree_demanda").select("descargado", F.explode("data").alias("j"))
ser = raw.select("descargado", F.explode("j.included").alias("s"))
val = ser.select("descargado", F.col("s.attributes.title").alias("serie"), F.explode("s.attributes.values").alias("v"))
df = val.select("descargado", "serie",
                F.to_date(F.substring("v.datetime", 1, 10)).alias("fecha"),
                F.col("v.value").cast("double").alias("valor_mwh"))

n_raw = df.count()
ok = df.filter(F.col("serie").isNotNull() & F.col("fecha").isNotNull() & F.col("valor_mwh").isNotNull()
               & (F.col("valor_mwh") > 0) & (F.col("fecha") <= F.lit(LIMITE_REE)))
final = con_particion(ultimo_valor(ok, ["serie", "fecha"]))
upsert(final, "ree_demanda_diaria", ["serie", "fecha"], "ree_demanda", n_raw - ok.count())

In [ ]:
# ENTSOG: flujo físico diario de Enagás (el día gas empieza a las 06:00 CET; la fecha es la de periodFrom)
raw = leer_bronze("entsog_flujo_fisico").select("descargado", F.explode("data").alias("r"))
df = raw.select("descargado",
                F.to_date(F.substring("r.periodFrom", 1, 10)).alias("fecha"),
                F.col("r.pointKey").alias("point_key"),
                F.col("r.pointLabel").alias("point_label"),
                F.col("r.directionKey").alias("direction_key"),
                F.col("r.value").cast("double").alias("valor"),
                F.col("r.unit").alias("unidad"))

n_raw = df.count()
ok = df.filter(F.col("fecha").isNotNull() & F.col("point_key").isNotNull() & F.col("direction_key").isNotNull()
               & F.col("valor").isNotNull() & (F.col("valor") >= 0) & (F.col("fecha") <= F.lit(LIMITE_ENTSOG)))
final = con_particion(ultimo_valor(ok, ["fecha", "point_key", "direction_key"]))
upsert(final, "entsog_flujo_fisico", ["fecha", "point_key", "direction_key"], "entsog_flujo_fisico", n_raw - ok.count())

In [ ]:
# AGSI+: almacenamiento subterráneo de España (los valores llegan como texto)
raw = leer_bronze("agsi_almacenamiento").select("descargado", F.explode("data").alias("r"))
existentes = raw.select("r.*").columns
MAPA = {"gasInStorage": "gas_in_storage_twh", "full": "full_pct", "trend": "trend_pct",
        "injection": "injection_gwh", "withdrawal": "withdrawal_gwh",
        "workingGasVolume": "working_gas_volume_twh",
        "injectionCapacity": "injection_capacity_gwh", "withdrawalCapacity": "withdrawal_capacity_gwh"}
cols = [(F.col(f"r.{o}").cast("double") if o in existentes else F.lit(None).cast("double")).alias(n) for o, n in MAPA.items()]
df = raw.select("descargado", F.to_date("r.gasDayStart").alias("fecha"), *cols)

n_raw = df.count()
ok = df.filter(F.col("fecha").isNotNull() & F.col("gas_in_storage_twh").isNotNull() & (F.col("gas_in_storage_twh") >= 0))
final = con_particion(ultimo_valor(ok, ["fecha"]))
upsert(final, "agsi_almacenamiento", ["fecha"], "agsi_almacenamiento", n_raw - ok.count())

In [ ]:
# MITECO: reserva por embalse y semana (hm³). Los números llegan como texto con coma decimal ('91,00').
def a_num(c):
    s = F.trim(F.col(c).cast("string"))
    # con coma: los puntos son separador de miles y la coma el decimal; sin coma: se deja tal cual
    return (F.when(s.contains(","), F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", "."))
             .otherwise(s).cast("double"))

raw = leer_bronze("miteco_embalses").select("descargado", F.explode("data").alias("r"))
df = (raw.select("descargado", "r.*")
         .select("descargado",
                 F.to_date("fecha").alias("fecha"),
                 F.col("ambito"), F.col("embalse"),
                 a_num("agua_total").alias("capacidad_hm3"),
                 a_num("agua_actual").alias("reserva_hm3"),
                 F.regexp_replace(F.col("electrico").cast("string"), r"[.,].*$", "").cast("int").alias("es_electrico")))

n_raw = df.count()
ok = df.filter(F.col("fecha").isNotNull() & F.col("ambito").isNotNull() & F.col("embalse").isNotNull()
               & (F.col("capacidad_hm3") > 0) & (F.col("reserva_hm3") >= 0))
print(f"MITECO: {n_raw} filas en Bronze, {ok.count()} válidas")
final = con_particion(ultimo_valor(ok, ["ambito", "embalse", "fecha"]))
upsert(final, "miteco_embalses_semanal", ["ambito", "embalse", "fecha"], "miteco_embalses", n_raw - ok.count())

In [ ]:
# Validaciones finales: tabla no vacía y ninguna clave natural repetida
fallos = []
for t, claves in [("ree_demanda_diaria", ["serie", "fecha"]),
                  ("entsog_flujo_fisico", ["fecha", "point_key", "direction_key"]),
                  ("agsi_almacenamiento", ["fecha"])]:
    d = spark.table(f"{SCHEMA}.{t}")
    n, distintas = d.count(), d.select(*claves).distinct().count()
    r = d.agg(F.min("fecha").alias("desde"), F.max("fecha").alias("hasta")).first()
    print(f"{t}: {n} filas | {r['desde']} -> {r['hasta']}")
    if n == 0: fallos.append(f"{t} vacía")
    if n != distintas: fallos.append(f"{t} con claves repetidas ({n - distintas})")

display(spark.createDataFrame(resumen))
if fallos:
    raise RuntimeError(f"Validaciones Silver fallidas: {fallos}")